<a href="https://colab.research.google.com/github/TurkuNLP/intro-to-nlp/blob/master/ex9_word2vec_solved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.8 MB/s eta 0:00:00


In [1]:
!wget http://dl.turkunlp.org/intro-to-nlp/enwiki-20220301-sample-tokenized.txt

--2026-04-15 19:42:16--  http://dl.turkunlp.org/intro-to-nlp/enwiki-20220301-sample-tokenized.txt
Resolving dl.turkunlp.org (dl.turkunlp.org)... 195.148.30.23
Connecting to dl.turkunlp.org (dl.turkunlp.org)|195.148.30.23|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 543383248 (518M) [text/plain]
Saving to: ‘enwiki-20220301-sample-tokenized.txt’

enwiki-20220301-sam 100%[===================>] 518.21M  11.0MB/s    in 40s     

2026-04-15 19:42:57 (13.0 MB/s) - ‘enwiki-20220301-sample-tokenized.txt’ saved [543383248/543383248]



The dataset contains 3.8 million sentences and 102 million tokens. Training word2vec on full data and 5 epochs takes about 22 minutes. Trim the data while debugging.

In [6]:
sentences = []
with open("enwiki-20220301-sample-tokenized.txt", "r") as f:
    for line in f:
        sentences.append(line.split())
print(f"Number of sentences: {len(sentences)}")
print(f"Number of tokens: {sum([len(s) for s in sentences])}")

Number of sentences: 3869427
Number of tokens: 102401825


In [7]:
%%time

# train model
model = Word2Vec(
    sentences,
    sg=0,
    vector_size=40,
    window=2,
    min_count=1,
    workers=1,
    epochs=5
)

CPU times: user 22min 18s, sys: 6.42 s, total: 22min 24s
Wall time: 22min 26s


In [11]:
pred = model.predict_output_word(["Helsinki", "is", "of", "Finland"], topn=5)
for w,s in pred:
    print(f"{w}: {s:.6f}")

region: 0.000024
Biennale: 0.000023
University: 0.000020
part: 0.000020
headquartered: 0.000017


Maybe not great, but these are still understandable and not random.

In [12]:
for word in ['good', 'bad', 'dog', 'cat', 'king', 'queen']:
  print(f'{word}:\t', [w for w, s in model.wv.most_similar(word)])
  print()

good:	 ['decent', 'bad', 'little', 'great', 'fair', 'real', 'nice', 'genuine', 'hard', 'big']

bad:	 ['good', 'tough', 'terrible', 'little', 'strange', 'decent', 'quiet', 'dangerous', 'crazy', 'horrible']

dog:	 ['cat', 'donkey', 'cow', 'baby', 'horse', 'monster', 'rabbit', 'creature', 'mask', 'boy']

cat:	 ['dog', 'rabbit', 'monkey', 'goat', 'snake', 'donkey', 'cow', 'wolf', 'monster', 'spider']

king:	 ['prince', 'emperor', 'monarch', 'queen', 'princess', 'sultan', 'duke', 'throne', 'kingdom', 'Emperor']

queen:	 ['king', 'princess', 'prince', 'knight', 'monarch', 'bride', 'consort', 'empress', 'demon', 'warrior']



Not too bad a result for a half-hour run! Some observations:

* Perhaps surprisingly, antonyms (words with opposite meaning) show up in the lists of most similar words. This is a known issue with word embeddings (see e.g. Scheible et al. 2013) and makes sense if you think about it: for example, bad can appear in many of the same contexts as good.
* These word embeddings group animal words with both dog and cat, but don't show the level of granularity that can be get with the embeddings trained on more data, where different types of dogs and cats were more similar to the words dog and cat (respectively) than other animals.
* We see a similar phenomenon with king and queen as with animals, with many words related to royalty and ruling classes showing up in both, with more limited distinction by e.g. gender than is often found in embeddings trained on more data.


In [5]:
# word2vec TOY EXAMPLE
# NOTE: This is a toy model that only demonstrates how to train word2vec,
# and that it will learn very simple patterns.
# This model will not give reasonable embedding similarities because of lack of data.

from gensim.models import Word2Vec

sentence_a = ["the", "cat", "liked", "milk"]
sentence_b = ["the", "dog", "ate", "food"]

sentences = [sentence_a]*7000 + [sentence_b]*7000 # repeat the same toy sentences multiple times

# train model
model = Word2Vec(
    sentences,
    sg=0,
    vector_size=40,
    window=2,
    min_count=1,
    workers=1,
    epochs=100
)

# test target word prediction (language modeling objective)
for context in [["the", "liked", "milk"], ["the", "ate", "food"], ["the", "liked", "food"]]:
    print("Context:", context)
    pred = model.predict_output_word(context, topn=5)
    print("Top predictions:")
    for w,s in pred:
        print(f"{w}: {s:.2f}")
    print()

# the model is able to learn a very simple pattern, where "liked milk" predicts "cat", and "ate food" predicts "dog"
# For "liked food" context, dog/cat/milk/ate are equally likely

Context: ['the', 'liked', 'milk']
Top predictions:
cat: 0.68
milk: 0.11
liked: 0.10
the: 0.06
dog: 0.02

Context: ['the', 'ate', 'food']
Top predictions:
dog: 0.72
food: 0.09
ate: 0.09
the: 0.06
cat: 0.01

Context: ['the', 'liked', 'food']
Top predictions:
dog: 0.22
milk: 0.21
cat: 0.20
ate: 0.20
the: 0.13

